# Загрузка данных TomskFluxNet

Конфигурации включают площадки подготовки L0 и отдельные дополнительные приборы. Загружаются только
отсутствующие локальные файлы. Год и месяц берутся из открываемого каталога
и проверяются по **всем** значениям колонки `data` перед сохранением в `raw`.
Для начала 20-минутного интервала допускается метка ровно за 20 минут
до начала месяца. Такие записи отдельно учитываются в отчёте.
Несоответствия сохраняются в `_temp`, имеющиеся CSV не перезаписываются.

`Oblkom_down` — отдельный прибор с собственным каталогом; его данные
не объединяются с `Oblkom_30m`.


In [ ]:
from collections import Counter
from datetime import datetime, timedelta, timezone
import hashlib
import io
import json
import os
from pathlib import Path
import shutil
from urllib.parse import unquote, urljoin, urlsplit

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from bs4 import BeautifulSoup
from tqdm import tqdm

import l0_data
from l0_data import RAW_ROOT, TOMSK_SITES

http_session = requests.Session()
http_session.mount('https://', HTTPAdapter(max_retries=Retry(
    total=2, backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=['GET', 'HEAD'],
)))


In [ ]:
load_configs = [
    {'site': 'BEK_30m',       'server_dirs': ('bek',),           'month_suffix': 'bek'},
    {'site': 'Oblkom_30m',    'server_dirs': ('oblkom',),        'month_suffix': 'oblkom'},
    {'site': 'Oblkom_down', 'server_dirs': ('oblkom',),      'month_suffix': 'oblkom_down'},  # Отдельный прибор.
    {'site': 'Voronino_10m',  'server_dirs': ('voronino',),      'month_suffix': 'voronino'},
    {'site': 'IMCES_40m',     'server_dirs': ('imces', 'bdmk'),  'month_suffix': 'bdmk40'},
    {'site': 'IMCES_30m',     'server_dirs': ('imces', 'bdmk'),  'month_suffix': 'bdmk31'},  # Имя площадки L0 сохранено; на сервере в 2026 г. уровень 31 м.
    {'site': 'IMCESmezo_30m', 'server_dirs': ('imces', 'mezo'),  'month_suffix': 'mezo'},
]
configured_sites = [item['site'] for item in load_configs]
assert len(configured_sites) == len(set(configured_sites)), 'Повторяющиеся локальные имена приборов'
assert set(TOMSK_SITES).issubset(configured_sites)

years = [2026]
months = range(1, 13)
keywords = ['moment', '_py']
BASE_URL = 'https://mirror.tomskfluxnet.ru/TomskFluxNet/'
save_dir = RAW_ROOT / 'Tomsk_py'
DOWNLOAD = True  # False: только поиск ссылок без загрузки CSV.

report_root = Path(l0_data.__file__).resolve().parent / '_temp'
moscow_tz = timezone(timedelta(hours=3))


In [ ]:
def get_files_from_html_index(url, checks):
    """Читает HTML-каталог, сохраняя результат запроса в checks."""
    if not urlsplit(url).path.endswith('/'):
        raise ValueError(f'Ожидался URL каталога: {url}')
    try:
        with http_session.get(url, timeout=(15, 45)) as response:
            checks.append({'url': url, 'status': response.status_code})
            if response.status_code == 404:
                return []
            response.raise_for_status()
            if 'html' not in response.headers.get('Content-Type', '').lower():
                raise ValueError(f'Ожидался HTML-каталог: {url}')
            base = urlsplit(response.url)
            soup = BeautifulSoup(response.text, 'html.parser')
            files = set()
            for link in soup.find_all('a', href=True):
                href = link['href']
                if href.startswith(('?', '#')):
                    continue
                candidate = urljoin(response.url, href)
                parsed = urlsplit(candidate)
                if (parsed.scheme == base.scheme and parsed.netloc == base.netloc
                        and parsed.path.startswith(base.path)
                        and parsed.path.lower().endswith('.csv')):
                    files.add(candidate)
            return sorted(files)
    except (requests.RequestException, ValueError) as error:
        checks.append({'url': url, 'error': str(error)})
        raise


def discover_files(configs, years, months):
    """Сохраняет ожидаемый период из каталога, а не из имени CSV."""
    records, checks, errors = [], [], []
    for config in configs:
        for year in years:
            for month in months:
                relative = '/'.join([
                    str(year), *config['server_dirs'],
                    f"{month:02d}_{year}_{config['month_suffix']}", '',
                ])
                parent_folder = urljoin(BASE_URL, relative)
                candidates = {}
                for folder in (urljoin(parent_folder, 'py/'), parent_folder):
                    try:
                        links = get_files_from_html_index(folder, checks)
                    except (requests.RequestException, ValueError) as error:
                        errors.append({'site': config['site'], 'year': year,
                                       'month': month, 'url': folder, 'error': str(error)})
                        continue
                    for file_url in links:
                        name = unquote(urlsplit(file_url).path.rsplit('/', 1)[-1])
                        if Path(name).name != name:
                            raise ValueError(f'Некорректное имя файла: {name}')
                        if all(keyword in name for keyword in keywords):
                            candidates.setdefault(name, file_url)
                for name, file_url in sorted(candidates.items()):
                    destination = save_dir / config['site'] / name
                    records.append({
                        'site': config['site'], 'year': year, 'month': month,
                        'folder_url': parent_folder, 'url': file_url,
                        'filename': name, 'path': str(destination),
                        'exists_locally': destination.is_file(),
                    })
        print(config['site'], sum(item['site'] == config['site'] for item in records),
              'файлов на сервере', flush=True)
    # Один путь назначения не должен соответствовать разным месяцам.
    counts = Counter(item['path'] for item in records)
    conflicts = [item for item in records if counts[item['path']] > 1]
    if conflicts:
        errors.extend(dict(item, error='Одинаковый путь назначения для разных каталогов')
                      for item in conflicts)
        records = [item for item in records if counts[item['path']] == 1]
    return records, checks, errors


In [ ]:
class DateValidationError(ValueError):
    def __init__(self, message, details):
        super().__init__(message)
        self.details = details


def validate_csv_dates(content, expected_year, expected_month):
    """Проверяет каждую метку data; временные метки и данные не изменяет."""
    details = {'expected_period': f'{expected_year:04d}-{expected_month:02d}'}
    try:
        frame = pd.read_csv(io.BytesIO(content), sep=';')
    except Exception as error:
        raise DateValidationError(f'CSV не читается: {error}', details) from error
    if 'data' not in frame.columns or frame.empty:
        raise DateValidationError('Нет колонки data или нет строк данных', details)
    # Сравниваем календарь, записанный прибором. Смещение UTC здесь
    # не применяется: перевод времени выполняется в подготовке L0.
    text = frame['data'].astype('string').str.strip().str.replace(
        r'(?:Z|[+-]\d{2}:?\d{2})$', '', regex=True,
    )
    dates = pd.to_datetime(text, format='mixed', errors='coerce')
    invalid = dates.isna()
    period_counts = dates.dropna().dt.strftime('%Y-%m').value_counts().to_dict()
    details.update(rows=len(frame), invalid_dates=int(invalid.sum()),
                   actual_periods={key: int(value) for key, value in period_counts.items()})
    if invalid.any():
        details['invalid_examples'] = frame.loc[invalid, 'data'].astype(str).head(5).tolist()
        raise DateValidationError('Есть неразбираемые или пустые даты', details)
    start = pd.Timestamp(expected_year, expected_month, 1)
    end = start + pd.offsets.MonthBegin(1)
    inside = (dates >= start) & (dates < end)
    boundary = dates == start - pd.Timedelta(minutes=20)
    outside = ~(inside | boundary)
    details.update(
        first_time=str(dates.min()), last_time=str(dates.max()),
        rows_in_expected_month=int(inside.sum()), boundary_rows=int(boundary.sum()),
        unexpected_rows=int(outside.sum()), duplicate_timestamps=int(dates.duplicated().sum()),
    )
    if outside.any() or not inside.any():
        details['unexpected_examples'] = frame.loc[outside, 'data'].astype(str).head(5).tolist()
        raise DateValidationError('Год/месяц в data не соответствует каталогу', details)
    return details


In [ ]:
def download_csv(record, run_dir):
    """Проверяет существующий CSV либо загружает и проверяет новый."""
    destination = Path(record['path'])
    result = dict(record)
    result['status'] = 'started'
    if destination.exists():
        try:
            content = destination.read_bytes()
            result['date_check'] = validate_csv_dates(content, record['year'], record['month'])
            result.update(status='already_present', bytes=len(content),
                          sha256=hashlib.sha256(content).hexdigest())
        except DateValidationError as error:
            result.update(status='existing_invalid_dates', error=str(error), date_check=error.details)
        except OSError as error:
            result.update(status='local_read_error', error=str(error))
        return result

    staging = run_dir / 'staging' / record['site'] / record['filename']
    staging.parent.mkdir(parents=True, exist_ok=True)
    try:
        with http_session.get(record['url'], stream=True, timeout=(15, 90)) as response:
            response.raise_for_status()
            result['last_modified'] = response.headers.get('Last-Modified')
            with staging.open('xb') as stream:
                for block in response.iter_content(1024 * 1024):
                    stream.write(block)
        content = staging.read_bytes()
        result.update(bytes=len(content), sha256=hashlib.sha256(content).hexdigest())
        try:
            result['date_check'] = validate_csv_dates(content, record['year'], record['month'])
        except DateValidationError as error:
            rejected = run_dir / 'rejected' / record['site'] / record['filename']
            rejected.parent.mkdir(parents=True, exist_ok=True)
            staging.rename(rejected)
            result.update(status='rejected_dates', error=str(error),
                          date_check=error.details, rejected_path=str(rejected))
            return result

        # Режим xb запрещает перезапись даже при появлении файла после поиска.
        destination.parent.mkdir(parents=True, exist_ok=True)
        created = False
        try:
            with destination.open('xb') as output:
                created = True
                with staging.open('rb') as source:
                    shutil.copyfileobj(source, output)
                output.flush()
                os.fsync(output.fileno())
            if hashlib.sha256(destination.read_bytes()).hexdigest() != result['sha256']:
                raise OSError('Контрольная сумма записанного файла не совпадает')
        except Exception:
            if created:
                destination.unlink(missing_ok=True)
            raise
        staging.unlink()
        result['status'] = 'downloaded'
    except FileExistsError as error:
        result.update(status='destination_or_staging_exists', error=str(error))
    except requests.RequestException as error:
        result.update(status='network_error', error=str(error))
    except OSError as error:
        result.update(status='write_error', error=str(error))
    return result


In [ ]:
run_dir = report_root / ('tomsk_download_' + datetime.now(moscow_tz).strftime('%Y%m%d_%H%M%S'))
run_dir.mkdir(parents=True, exist_ok=False)
(report_root / 'latest_tomsk_download.json').write_text(
    json.dumps({'run_directory': str(run_dir)}, ensure_ascii=False, indent=2), encoding='utf-8',
)

files_to_download, index_checks, discovery_errors = discover_files(load_configs, years, months)
availability_summary = []
for config in load_configs:
    selected = [item for item in files_to_download if item['site'] == config['site']]
    existing = sum(item['exists_locally'] for item in selected)
    availability_summary.append({'site': config['site'], 'remote_files': len(selected),
                                 'local_files': existing, 'new_files': len(selected) - existing})
(run_dir / 'inventory.json').write_text(json.dumps({
    'configs': load_configs, 'years': years, 'files': files_to_download,
    'indexes': index_checks, 'discovery_errors': discovery_errors,
    'summary': availability_summary,
}, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(availability_summary))
print('Ошибок поиска:', len(discovery_errors))


In [ ]:
download_results = []
if DOWNLOAD:
    with (run_dir / 'results.jsonl').open('w', encoding='utf-8') as journal:
        for record in tqdm(files_to_download, desc='Проверка и загрузка'):
            result = download_csv(record, run_dir)
            download_results.append(result)
            journal.write(json.dumps(result, ensure_ascii=False) + '\n')
            journal.flush()
            print(result['site'], result['filename'], result['status'], flush=True)
else:
    print('Скачивание выключено (DOWNLOAD = False)')

download_summary = []
for config in load_configs:
    results = [item for item in download_results if item['site'] == config['site']]
    counts = Counter(item['status'] for item in results)
    download_summary.append({'site': config['site'], 'remote_files': len(results),
                             'downloaded': counts['downloaded'],
                             'already_present': counts['already_present'],
                             'rejected_dates': counts['rejected_dates'] + counts['existing_invalid_dates'],
                             'other_errors': sum(value for key, value in counts.items()
                                                 if key not in {'downloaded', 'already_present',
                                                                'rejected_dates', 'existing_invalid_dates'})})
report = {
    'checked_at_moscow': datetime.now(moscow_tz).isoformat(), 'base_url': BASE_URL,
    'save_dir': str(save_dir), 'run_directory': str(run_dir), 'years': years,
    'date_policy': 'Every data timestamp must be in the page month; the exact 20-minute leading boundary is allowed.',
    'download_enabled': DOWNLOAD, 'discovery_errors': discovery_errors,
    'summary': download_summary, 'results': download_results,
}
(run_dir / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
display(pd.DataFrame(download_summary))
print('Отчёт:', run_dir / 'report.json')


Для поиска без скачивания установите `DOWNLOAD = False`. Файлы, уже лежащие
в `raw/Tomsk_py/<site>`, проверяются по датам, но не перезаписываются.
Временные файлы, отчёты и отклонённые CSV находятся в `_temp/tomsk_download_*`.
`results.jsonl` сохраняется после каждого файла, поэтому результаты доступны
и при прерывании выполнения. Для повторной попытки запустите поиск заново:
уже сохранённые файлы будут пропущены.
